# 02 · RAGAS 평가

Phase 4 — RAG 파이프라인을 **골든셋 50개**로 실행하고 **Claude(Haiku 4.5)** 를 평가자로 4개 지표를 측정한다.

- `faithfulness` — 답변이 검색된 문맥에 충실한가 (환각 여부)
- `answer_relevancy` — 답변이 질문에 적절한가
- `context_precision` — 검색된 문맥 중 정답에 기여하는 비율
- `context_recall` — 정답을 뒷받침하는 문맥이 얼마나 검색되었는가

주요 로직은 `src/evaluator.py`로 분리했다 — **RAGAS×langchain 1.x 호환 셰임도 `evaluator` import 시 자동 적용**된다.

### 사전 준비
1. pgvector 테이블 적재 — `01_pipeline.ipynb` 또는 `python scripts/index_corpus.py`
2. Ollama 구동 (`gemma4:e2b`)
3. `.env`에 `ANTHROPIC_API_KEY`

> 전체 파이프라인을 한 번에 돌리려면: `python scripts/run_ragas_eval.py --collect`

## 1. 설정 & 임포트

`evaluator`를 import하는 순간 호환 셰임이 적용된다 (ragas import보다 먼저).

In [ ]:
import os

from src import config, dataset, embedder, evaluator, rag, retriever

golden_set = dataset.load_golden_set()
assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY가 .env에 없습니다.'
print(f'골든셋: {len(golden_set)}개 / 평가자: {config.JUDGE_MODEL}')

## 2. RAG 파이프라인 재구성 (기존 테이블 연결)

이미 적재된 pgvector 테이블에 **연결만** 한다. `init_table`을 호출하면 데이터가 지워지므로 호출하지 않는다.

In [ ]:
embeddings = embedder.build_embeddings()
engine = retriever.get_engine()
store = retriever.get_store(engine, embeddings)  # 재생성 X, 연결만

probe = store.similarity_search(golden_set[0]['question'], k=1)
assert probe, 'pgvector 테이블이 비어 있습니다. 01_pipeline 또는 scripts/index_corpus.py로 먼저 적재하세요.'

retr = retriever.get_retriever(store)
qa_chain = rag.build_qa_chain(retr, rag.build_llm())
print(f'RAG 체인 재구성 완료 — 샘플 검색 {len(probe)}건')

## 3. 골든셋으로 RAG 실행 → 평가 입력 수집 (⏳ 무거움, 캐싱됨)

골든셋 50개를 RAG에 통과시켜 `(user_input, retrieved_contexts, response, reference)`를 모은다. 결과는 `results/eval_inputs.json`에 캐싱된다 (`FORCE_RECOLLECT=True`로 재수집).

In [ ]:
FORCE_RECOLLECT = False

if config.EVAL_INPUTS_PATH.exists() and not FORCE_RECOLLECT:
    records = evaluator.load_eval_inputs()
    print(f'캐시 로드: {len(records)}개 ({config.EVAL_INPUTS_PATH})')
else:
    records = evaluator.collect_eval_inputs(qa_chain, golden_set)  # tqdm 진행도
    evaluator.save_eval_inputs(records)
    print(f'수집 완료 & 저장: {len(records)}개')

_s = records[0]
print('\n[샘플 0]')
print('  질문 :', _s['user_input'])
print('  답변 :', _s['response'][:120])
print('  정답 :', _s['reference'])
print('  문맥 수:', len(_s['retrieved_contexts']))

## 4. RAGAS 평가자 설정

평가자 LLM은 Claude Haiku 4.5(`temperature=0`), 임베딩은 RAG와 동일한 BGE-m3-ko를 래핑한다.

In [ ]:
judge = evaluator.build_judge()
evaluator_llm, evaluator_emb = evaluator.build_evaluators(judge, embeddings)
print('평가자 설정 완료:', config.JUDGE_MODEL)

## 5. EvaluationDataset 구성 & 4개 지표 측정 (⏳ Claude API 호출)

`max_workers`를 낮춰 API rate limit을 완화한다.

In [ ]:
eval_dataset = evaluator.build_dataset(records)
print(f'EvaluationDataset: {len(eval_dataset)}개 샘플')

result = evaluator.run_ragas(eval_dataset, evaluator_llm, evaluator_emb, max_workers=4)
print(result)

## 6. 결과 저장 & 집계

샘플별 점수 → `results/ragas_baseline_detail.csv`, 지표별 평균 → `results/ragas_baseline.json`.

In [ ]:
aggregate = evaluator.save_results(result)
print('=== 지표별 평균 점수 ===')
for k, v in aggregate.items():
    print(f'  {k:20s}: {v:.4f}')
print(f'\n저장: {config.RAGAS_RESULT_PATH}\n      {config.RAGAS_DETAIL_PATH}')

## 7. 지표 해석 메모

점수는 모두 **0~1**(높을수록 좋음).

- **faithfulness** — 낮으면 문맥에 없는 내용을 지어냄(환각). 로컬 소형 LLM(gemma4:e2b)에서 종종 낮다.
- **answer_relevancy** — 낮으면 질문과 동떨어진/장황한 답변.
- **context_precision** — top-k 중 정답 기여 청크 비율. 낮으면 리트리버가 노이즈를 많이 가져옴.
- **context_recall** — 정답 근거가 검색에 얼마나 포함됐는가. **Phase 5 벡터DB/임베딩 비교의 핵심 지표.**

### ⚠️ MAX_DOCS 영향
일부 문서만 색인했다면 골든셋 정답 문서가 색인에 없어 `context_recall`이 구조적으로 낮게 나온다. 제대로 된 baseline은 `python scripts/index_corpus.py`(전체 색인) 후 `FORCE_RECOLLECT=True`로 재수집해 평가한다.

이 baseline이 **Phase 5(벡터DB·임베딩 비교)** 의 기준선이 된다.